    # Lab 03 · MNIST with a Convolutional Neural Network
    **AI Lab · University of Technology Bahrain** · ⏱ ~15 min · ⚡ GPU

    **You will learn**
    - Convolution, pooling and feature maps
- Why CNNs beat MLPs on images
- Visualising what the filters detect

    > **First time on Kaggle?** Make sure your account is phone-verified, then in the right-hand panel open
    > **Settings → Accelerator** (choose *GPU T4 x2* when the lab needs it) and **Settings → Internet → On**.
    > Run cells top to bottom with **Shift + Enter**.

In [ ]:
# ⚙️ Setup: run me first
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

ON_KAGGLE = os.path.exists("/kaggle/working")
OUT_DIR = "/kaggle/working" if ON_KAGGLE else "."
gpus = tf.config.list_physical_devices("GPU")
print("Running on Kaggle:", ON_KAGGLE)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", [g.name for g in gpus] if gpus else "none (CPU). Turn on Settings > Accelerator > GPU if this lab needs it")
keras.utils.set_random_seed(42)

In [ ]:
# 📦 Dataset loader: downloads with Keras, falls back to datasets attached via "+ Add Input"
def load_dataset(name):
    """name: 'mnist', 'fashion_mnist' or 'cifar10' -> (x_train, y_train), (x_test, y_test)"""
    try:
        return getattr(keras.datasets, name).load_data()
    except Exception as err:
        if name == "mnist":  # Kaggle-attached copy of mnist.npz, if any
            hits = glob.glob("/kaggle/input/**/mnist.npz", recursive=True)
            if hits:
                d = np.load(hits[0])
                return (d["x_train"], d["y_train"]), (d["x_test"], d["y_test"])
        raise RuntimeError(
            "Could not download the dataset. On Kaggle: right panel > Settings > Internet > On "
            "(needs a phone-verified account), then run this cell again.") from err

In [ ]:
(x_train, y_train), (x_test, y_test) = load_dataset("mnist")
x_train = (x_train.astype("float32") / 255.0)[..., None]   # add channel axis -> (N, 28, 28, 1)
x_test = (x_test.astype("float32") / 255.0)[..., None]
print(x_train.shape)

## 1 · A small CNN
Watch the **Output Shape** column in the summary: the image shrinks while the number of channels grows.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation="relu", name="conv1"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", name="conv2"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
t = time.time()
history = model.fit(x_train, y_train, epochs=6, batch_size=128, validation_split=0.1, verbose=2)
print(f"Training time: {time.time()-t:.1f}s")
print("Test accuracy:", round(model.evaluate(x_test, y_test, verbose=0)[1], 4))

In [ ]:
# 📈 Learning curves
def plot_history(h):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(h.history["loss"], label="train"); ax[0].plot(h.history["val_loss"], label="validation")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h.history["accuracy"], label="train"); ax[1].plot(h.history["val_accuracy"], label="validation")
    ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()
plot_history(history)

## 2 · What do the filters see?

In [ ]:
# The 32 learned 3x3 kernels of the first layer
w = model.get_layer("conv1").get_weights()[0][:, :, 0, :]
fig, axes = plt.subplots(2, 16, figsize=(14, 2))
for i, ax in enumerate(axes.flat):
    ax.imshow(w[:, :, i], cmap="gray"); ax.axis("off")
plt.suptitle("First-layer filters"); plt.show()

# Feature maps for one digit
probe = keras.Model(model.inputs, model.get_layer("conv1").output)
maps = probe.predict(x_test[:1], verbose=0)[0]
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(maps[:, :, i], cmap="viridis"); ax.axis("off")
plt.suptitle(f"Feature maps for a '{y_test[0]}'"); plt.show()

## 🧪 Try it
1. Compare with Lab 02: how many **parameters** does each model have, and which is more accurate?
2. Add a third `Conv2D(128, 3)` layer. What happens to the output shape, and why might it fail?
3. Draw your own digit: make a 28×28 array with a stroke of 1s and ask `model.predict`.

### 💾 Save your work
Click **Save Version** (top right) → *Quick Save* to keep your edits, or *Save & Run All* to store the outputs too.
Files written to `/kaggle/working` appear under **Output** and can be downloaded.